# Week 5: Deep Learning and Global Models
## In-Class Exercises

**Objective.** Fit a neural forecaster, understand what "global" buys you, and find the point where the extra machinery stops paying for itself.

> **Before you start:** switch the Colab runtime to a GPU (Runtime -> Change runtime type -> T4). CPU works too, just slower. The install cell takes 2-3 minutes, so start it now.

### How this notebook works

Three parts, each building on the one before it.

| Part | Format | Content |
| --- | --- | --- |
| 1 | Walkthrough | NHITS on a single series, and why the result is underwhelming. |
| 2 | Blanks we fill in together | A panel of 60 series: local vs. global for LightGBM, NHITS, and NBEATS. |
| 3 | On your own, ~15 min | Change the architecture and the input window, and measure what moved. |


In [ ]:
!pip install -q neuralforecast statsforecast mlforecast lightgbm plotly


In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, NBEATS
from neuralforecast.utils import AirPassengersDF
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, SeasonalNaive

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"   # try "plotly_dark", "ggplot2", "simple_white"


def mae(y, yhat):
    return np.mean(np.abs(np.asarray(y, float) - np.asarray(yhat, float)))

def mase(y, yhat, y_train, m=12):
    y_train = np.asarray(y_train, float)
    return mae(y, yhat) / np.mean(np.abs(y_train[m:] - y_train[:-m]))

AirPassengersDF.head()


---
## Part 1. NHITS on one short series

The Nixtla ecosystem uses one long format everywhere: `unique_id`, `ds`, `y`, one row per series per timestamp. Learn it once and every library in the stack accepts your data.

NHITS is a stack of MLP blocks at different sampling rates, so each block specializes in a frequency band. Fast, strong on benchmarks, and about to struggle on the 144-point series below. Watch the comparison against `AutoETS`.


In [ ]:
h, m = 12, 12
Y = AirPassengersDF.copy()
Y["ds"] = Y["ds"].dt.to_period("M").dt.to_timestamp()  # align to month-start, matching freq="MS" forecasts
Y_train = Y.iloc[:-h]
Y_test = Y.iloc[-h:]

nf = NeuralForecast(
    models=[NHITS(h=h, input_size=2 * h, max_steps=200, random_seed=1)],
    freq="MS",
)
nf.fit(df=Y_train)
nn_fc = nf.predict()
nn_fc.head()


In [ ]:
sf = StatsForecast(models=[AutoETS(season_length=m), SeasonalNaive(season_length=m)], freq="MS")
stat_fc = sf.forecast(df=Y_train, h=h)

merged = (Y_test.merge(nn_fc, on=["unique_id", "ds"])
                .merge(stat_fc, on=["unique_id", "ds"]))

tidy = pd.concat([
    Y_train.tail(48).assign(series="train").rename(columns={"y": "value"})[["ds", "series", "value"]],
    merged.melt(id_vars="ds", value_vars=["y", "NHITS", "AutoETS", "SeasonalNaive"],
                var_name="series", value_name="value"),
])
px.line(tidy, x="ds", y="value", color="series",
        title="One series, 144 observations").show()

for col in ["NHITS", "AutoETS", "SeasonalNaive"]:
    print(f"{col:14s} MASE: {mase(merged['y'], merged[col], Y_train['y'], m):.3f}")


**Notice:**

The network has thousands of parameters and 132 training observations. `AutoETS` has about a dozen parameters and a structural assumption that is true for this series. The small model with the right structure wins, which is neither a surprise nor a scandal.

Deep learning models are not "better forecasters." They **trade structural assumptions for data**. Give one 132 observations and it has nothing to trade. Give it 60 series of 132 observations and the trade pays, because a shared model learns the seasonal shape from all of them at once.

That is the argument for **global** models. Part 2 tests it - for three model families, not just one.

---
## Part 2. Sixty series, local vs. global

The panel below has three clusters of behavior plus short series with barely enough history to fit anything locally. Those short series are where global models earn their reputation.

**Local** means one model fit per series. **Global** means one model fit on the whole panel at once, sharing what it learns across series. We test both for `LightGBM`, `NHITS`, and `NBEATS`.


In [ ]:
def make_panel(n_per_group=20, n=72, seed=5):
    rng = np.random.default_rng(seed)
    rows = []
    start = pd.Timestamp("2015-01-01")
    for g, (trend, amp, noise) in enumerate([(0.4, 12, 3.0), (0.0, 25, 5.0), (-0.2, 5, 2.0)]):
        for i in range(n_per_group):
            uid = f"g{g}_s{i:02d}"
            # short series: the last third of each group has only 30 observations
            length = n if i < (2 * n_per_group) // 3 else 30
            t = np.arange(length)
            phase = rng.uniform(0, 2 * np.pi)
            level = rng.uniform(80, 160)
            y = (level + trend * t
                 + amp * np.sin(2 * np.pi * t / 12 + phase)
                 + rng.normal(scale=noise, size=length))
            rows.append(pd.DataFrame({
                "unique_id": uid,
                "ds": pd.date_range(start, periods=length, freq="MS"),
                "y": y,
            }))
    return pd.concat(rows, ignore_index=True)


panel = make_panel()
print(panel.groupby("unique_id").size().describe()[["count", "min", "max"]])

examples = panel[panel["unique_id"].isin(["g0_s00", "g1_s00", "g2_s19"])]
px.line(examples, x="ds", y="y", facet_col="unique_id", height=300,
        title="One series from each group").show()


`LightGBM` is cheap enough to fit locally on all 60 series in a few seconds - Beat 1 does that. Training a neural net from scratch is not: one NHITS fit takes about as long per series as it takes for the *whole panel* at once, because the cost is dominated by the number of gradient steps, not the amount of data. Looping that over 60 series would take the rest of class. Beat 2 fits NHITS and NBEATS locally on a small subset instead - `g0_s00`, `g1_s00`, `g2_s19`, the same three series plotted above - and globally on the full panel, so the comparison is still real, just not exhaustive.


In [ ]:
H = 12
panel_train = panel.groupby("unique_id", group_keys=False).apply(lambda d: d.iloc[:-H])
panel_test = panel.groupby("unique_id", group_keys=False).apply(lambda d: d.iloc[-H:])

# Local models: one fit per series.
sf = StatsForecast(models=[AutoETS(season_length=12), SeasonalNaive(season_length=12)], freq="MS", n_jobs=-1)
local_fc = sf.forecast(df=panel_train, h=H)
local_fc.head()


### Beat 1. LightGBM: local vs. global

Same recipe as Week 4's `mlforecast` pattern. Local: loop over `unique_id`, fit one `MLForecast` per series. Global: one `MLForecast` call on the whole panel - the lag and date features are computed per series either way, only the model-fitting loop changes.


In [ ]:
rows = []
for uid, g in panel_train.groupby("unique_id"):
    mlf = MLForecast(models={"LocalLightGBM": LGBMRegressor(n_estimators=100, verbosity=-1)},
                     freq="MS", lags=[1, 12], date_features=["month"])
    mlf.fit(g)
    rows.append(mlf.predict(H))
local_lgb_fc = pd.concat(rows, ignore_index=True)
local_lgb_fc.head()


In [ ]:
mlf_global = MLForecast(models={"GlobalLightGBM": LGBMRegressor(n_estimators=100, verbosity=-1)},
                        freq="MS", lags=[1, 12], date_features=["month"])
mlf_global.fit(panel_train)
global_lgb_fc = mlf_global.predict(H)
global_lgb_fc.head()


**Notice:** local LightGBM never wins on any series in this panel - check the win-rate column in the summary table below once we get there. The gap between local and global is present on both long and short series, but it is much larger on the short ones. A global fit pools the lag and date features across every series, so even a series with almost no history of its own still lands in a design matrix with plenty of rows.


### Beat 2. NHITS and NBEATS: local vs. global

Global first - the call is identical to Part 1, `NeuralForecast` infers the panel from `unique_id`, and one call trains both architectures.


In [ ]:
# TODO - fit ONE NeuralForecast across every series in the panel with BOTH NHITS and NBEATS,
#   and predict H steps.
#   Hint: the call is identical to Part 1, just two models in the list instead of one.
#   Use NHITS(h=H, input_size=2*H, max_steps=300, random_seed=1) and
#       NBEATS(h=H, input_size=2*H, max_steps=300, random_seed=1).
nf_global = ...
global_fc = ...


<details>
<summary><b>Show the lines</b></summary>

```python
nf_global = NeuralForecast(models=[NHITS(h=H, input_size=2 * H, max_steps=300, random_seed=1),
                                   NBEATS(h=H, input_size=2 * H, max_steps=300, random_seed=1)],
                           freq="MS")
nf_global.fit(df=panel_train)
global_fc = nf_global.predict()
```

Note what you did *not* write: no loop over series, no per-series hyperparameters, no special handling for the 30-observation series, and one `.fit()` call trained both architectures.
</details>


In [ ]:
subset_ids = ["g0_s00", "g1_s00", "g2_s19"]   # the same three series plotted above
subset_train = panel_train[panel_train["unique_id"].isin(subset_ids)]

short = subset_train[subset_train["unique_id"] == "g2_s19"]
print(f"{len(short)} training rows, input_size asks for {2 * H}")
for Model in [NHITS, NBEATS]:
    try:
        nf1 = NeuralForecast(models=[Model(h=H, input_size=2 * H, max_steps=100, random_seed=1)], freq="MS")
        nf1.fit(df=short)
    except Exception as e:
        print(f"{Model.__name__} FAILED: {e}")


`g2_s19` is one of the short series: 30 observations total, 18 left after holding out the test window, and `input_size=2*H=24` asks for a longer window than the series has. Local NHITS and NBEATS cannot even start training on it. `start_padding_enabled=True` pads the start of short series so a window of the requested length still exists.


In [ ]:
local_preds = {"NHITS": [], "NBEATS": []}
for Model in [NHITS, NBEATS]:
    name = Model.__name__
    for uid in subset_ids:
        one = subset_train[subset_train["unique_id"] == uid]
        nf1 = NeuralForecast(models=[Model(h=H, input_size=2 * H, max_steps=100, random_seed=1,
                                           start_padding_enabled=True)], freq="MS")
        nf1.fit(df=one)
        local_preds[name].append(nf1.predict())

local_nhits_fc = pd.concat(local_preds["NHITS"], ignore_index=True).rename(columns={"NHITS": "LocalNHITS"})
local_nbeats_fc = pd.concat(local_preds["NBEATS"], ignore_index=True).rename(columns={"NBEATS": "LocalNBEATS"})
local_nhits_fc.merge(local_nbeats_fc, on=["unique_id", "ds"]).head()


---
### Reading the results

Two tables. The first uses every series in the panel, for the methods cheap enough to run on all 60: the statistical baselines, LightGBM local and global, and the global neural nets. The second is restricted to the three-series subset, where local and global are directly comparable for all three model families.


In [ ]:
ev = (panel_test
      .merge(local_fc, on=["unique_id", "ds"])
      .merge(local_lgb_fc, on=["unique_id", "ds"])
      .merge(global_lgb_fc, on=["unique_id", "ds"])
      .merge(global_fc, on=["unique_id", "ds"]))

scales = (panel_train.groupby("unique_id")["y"]
          .apply(lambda s: np.mean(np.abs(s.to_numpy()[12:] - s.to_numpy()[:-12]))))

def per_series_mase(df, col):
    err = df.assign(e=lambda d: (d["y"] - d[col]).abs()).groupby("unique_id")["e"].mean()
    return err / scales

cols = ["AutoETS", "SeasonalNaive", "LocalLightGBM", "GlobalLightGBM", "NHITS", "NBEATS"]
per_series = pd.DataFrame({c: per_series_mase(ev, c) for c in cols})
lengths = panel_train.groupby("unique_id").size().rename("n_obs")

summary = pd.DataFrame({
    "mean MASE": per_series.mean(),
    "median MASE": per_series.median(),
    "worst 10% mean": per_series.apply(lambda c: c.nlargest(6).mean()),
    "win rate": per_series.idxmin(axis=1).value_counts(normalize=True).reindex(cols).fillna(0),
})
summary.round(3)


In [ ]:
# Long series have 60 training rows here (72 minus the 12-step holdout); short series have 18.
# Compare against the post-holdout count, not the pre-holdout one.
joined = per_series.join(lengths)
joined.groupby(joined["n_obs"] < 60).mean(numeric_only=True).rename(
    index={False: "long series", True: "short series"}).round(3)


In [ ]:
subset_test = panel_test[panel_test["unique_id"].isin(subset_ids)]
subset_ev = (subset_test
             .merge(local_lgb_fc, on=["unique_id", "ds"])
             .merge(global_lgb_fc, on=["unique_id", "ds"])
             .merge(global_fc.rename(columns={"NHITS": "GlobalNHITS", "NBEATS": "GlobalNBEATS"}),
                    on=["unique_id", "ds"])
             .merge(local_nhits_fc, on=["unique_id", "ds"])
             .merge(local_nbeats_fc, on=["unique_id", "ds"]))

subset_scales = scales.loc[subset_ids]

def subset_mase(col):
    err = subset_ev.assign(e=lambda d: (d["y"] - d[col]).abs()).groupby("unique_id")["e"].mean()
    return err / subset_scales

subset_cols = ["LocalLightGBM", "GlobalLightGBM", "LocalNHITS", "GlobalNHITS", "LocalNBEATS", "GlobalNBEATS"]
pd.DataFrame({c: subset_mase(c) for c in subset_cols}).round(3)


**Reading the full-panel table.**

1. Local LightGBM has the worst mean and never wins a single series - check the win-rate row. Global LightGBM is a clear improvement over it, everywhere, and costs almost nothing extra to fit.
2. On the long series, `AutoETS` is competitive or better, same as before: the structure it assumes is the structure this panel was generated with.
3. Mean and worst-decile tell different stories from the median for `NHITS`. A method with a mediocre mean and a bad tail is one you would not want to ship without knowing which series land in that tail - the tail is where the complaints come from, not the average.

**Reading the by-length table.**

The pattern from Part 1 shows up again for `AutoETS` and for LightGBM: both degrade sharply on short series, and global LightGBM recovers most of the ground local LightGBM loses. `NBEATS` barely notices the difference between long and short series, which is the strongest evidence in this notebook for the "trade structure for data" claim.

`NHITS` breaks the pattern. It does **not** clearly improve on short series here, and the subset table below shows why in one series: on `g2_s19`, the short series, local `NHITS` - trained from scratch on eighteen observations, with padding so it can train at all - beats global `NHITS` outright. That is not what the story predicts. Something about this architecture, this input size, or this seed is not generalizing to that series the way it does for `NBEATS`. Part 3 gives you a place to start finding out.

**Reading the subset table.** On the two long series, global beats local for all three families, as expected. On the short series, global beats local for LightGBM and NBEATS - and loses to local for NHITS. Three series is not enough to say this generalizes; it is enough to say the "global always wins" story is not automatic, and to know which architecture to distrust on this panel until you have investigated further.


---
## Part 3. Make it better, and say what moved

About 15 minutes. Turn one knob at a time; turning three at once teaches you nothing.

**Tasks.**

1. Re-fit the global NHITS with `input_size` of `H`, `2*H`, and `4*H`. Report the winner and whether it differs between long and short series. `input_size=2*H` is what struggled on the short series in Part 2 - see if a different value closes that gap.
2. Using `per_series` from Part 2, compute `LocalLightGBM - GlobalLightGBM` for every series, split by the same long/short grouping. Does LightGBM show the same "global wins on short series" pattern the neural nets showed, and if the size of the effect differs, say why you think that is.
3. Raise `max_steps` from 300 to 1000 on the best global NHITS configuration and plot MASE against training steps. Note where it stops improving and whether it degrades.
4. In two sentences: across all three families, which knob mattered most, and what would you try with another 30 minutes of compute?

Keep a results table. "I tried some things and the second one seemed better" is not a finding.


In [ ]:
# YOUR CODE HERE


<details>
<summary><b>Hints and what to expect</b></summary>

```python
results = []
for size_mult in [1, 2, 4]:
    nf_i = NeuralForecast(
        models=[NHITS(h=H, input_size=size_mult * H, max_steps=300, random_seed=1)], freq="MS")
    nf_i.fit(df=panel_train)
    f = nf_i.predict()
    e = panel_test.merge(f, on=["unique_id", "ds"])
    results.append({"input_size": size_mult * H,
                    "mean MASE": per_series_mase(e, "NHITS").mean()})
pd.DataFrame(results)
```

Expect `2*H` to be a reasonable default, and `4*H` to help long series while hurting short ones: a longer input window disqualifies short series from contributing training windows at all. That interaction between window length and series length never appears in tutorials and always appears in production.

For task 2:

```python
gap = (per_series["LocalLightGBM"] - per_series["GlobalLightGBM"]).to_frame("gap").join(lengths)
gap.groupby(gap["n_obs"] < 60).mean().rename(index={False: "long series", True: "short series"})
```
</details>


---
## Wrap-up

1. **Global models trade structure for data.** With one short series there is nothing to trade.
2. **The gain concentrates in short series and the error tail**, not the mean - and it shows up for LightGBM too, not just neural nets, though the mechanism differs: LightGBM pools rows for a shared feature space; NHITS and NBEATS learn a shared representation.
3. **The "global wins on short series" story is not automatic.** It held cleanly for LightGBM and NBEATS here and did not hold for NHITS. Run the comparison on your own panel before you trust it - the architecture matters, not just the local-vs-global choice.
4. **A short series can break local deep learning outright**, not just underperform it. `input_size` longer than the training history fails to fit at all, unless `start_padding_enabled=True`.
5. **`input_size` interacts with series length.** Longer windows silently discard short series.
6. **What "local" costs you is architecture-dependent.** Local LightGBM on the whole panel took seconds. Local neural nets did not scale past a handful of series in class time - that gap is itself a reason global models are the default for deep learning on panels, even before accuracy enters the argument.

Next week: probabilistic forecasts, and making a hierarchy add up.
